In [1]:
# 导入网络请求和数据处理工具
import requests
import pandas as pd
import json

# 1. 设置目标靶点：西安市的地理坐标 (北纬 34.34，东经 108.94)
# 我们直接请求 Open-Meteo 的底层 API 接口，这比解析 HTML 网页高效且合法
LATITUDE = 34.3416
LONGITUDE = 108.9398
START_DATE = "2025-01-01" # 抓取过去一年多的数据
END_DATE = "2026-03-30"

print(f"🚀 初始化爬虫... 目标城市: 西安 (Lat: {LATITUDE}, Lon: {LONGITUDE})")

# 2. 组装 API 请求 URL (获取空气质量和气象数据)
# hourly 参数代表我们要抓取每一小时的 PM2.5
url = f"https://air-quality-api.open-meteo.com/v1/air-quality?latitude={LATITUDE}&longitude={LONGITUDE}&start_date={START_DATE}&end_date={END_DATE}&hourly=pm2_5"

# 3. 发送网络请求，抓取数据 (这相当于你在浏览器按回车，但速度快一万倍)
print("📡 正在向服务器发送请求，请稍候...")
response = requests.get(url)

# 4. 检查是否被服务器拒绝 (状态码 200 代表成功)
if response.status_code == 200:
    print("✅ 数据抓取成功！正在解析 JSON 格式...")
    
    # 将原始的网络数据包转换成 Python 能看懂的字典
    raw_data = response.json()
    
    # 提取时间轴和 PM2.5 浓度
    timestamps = raw_data['hourly']['time']
    pm25_values = raw_data['hourly']['pm2_5']
    
    # 5. 使用 Pandas 将数据组装成结构化的二维表格 (DataFrame)
    df_pm25 = pd.DataFrame({
        'Date_Time': timestamps,
        'PM25_Target': pm25_values
    })
    
    # 简单清洗数据：去掉那些可能缺失的 NaN 脏数据
    df_pm25 = df_pm25.dropna()
    
    # 看看我们抓到了什么
    print(f"\n📊 成功组装表格！共抓取到 {len(df_pm25)} 条逐小时真实数据。")
    print(df_pm25.head()) # 打印前5行看看
    
    # 6. 保存战利品
    # 将抓取到的数据以 CSV 格式保存在当前文件夹
    filename = "xian_real_pm25_data.csv"
    df_pm25.to_csv(filename, index=False)
    print(f"\n💾 战利品已保存为：{filename}")
    print("你可以回到桌面的 pm25-predictive-modeling 文件夹里去验收了！")

else:
    print(f"❌ 抓取失败，服务器返回了状态码: {response.status_code}")
    print("可能的原因：网络连接问题或 API 限制。")

🚀 初始化爬虫... 目标城市: 西安 (Lat: 34.3416, Lon: 108.9398)
📡 正在向服务器发送请求，请稍候...
✅ 数据抓取成功！正在解析 JSON 格式...

📊 成功组装表格！共抓取到 10896 条逐小时真实数据。
          Date_Time  PM25_Target
0  2025-01-01T00:00        100.4
1  2025-01-01T01:00        103.4
2  2025-01-01T02:00         96.4
3  2025-01-01T03:00         67.5
4  2025-01-01T04:00         43.7

💾 战利品已保存为：xian_real_pm25_data.csv
你可以回到桌面的 pm25-predictive-modeling 文件夹里去验收了！
